# RLHF 完整三阶段 Pipeline

> InstructGPT / Llama2-Chat 的标准对齐流程。

## 背景
RLHF（Reinforcement Learning from Human Feedback）是 LLM 对齐的核心技术，通过三阶段将 base model 对齐到人类偏好。

## 三阶段
1. **SFT**：监督微调，用高质量对话数据 fine-tune base model
2. **RM**：训练奖励模型，用人类偏好对比数据
3. **PPO**：用 RM 奖励 + KL 罚在线训练 policy

## PPO 损失
L_total = L_PG - c_entropy * H(π) + c_vf * L_VF - β * KL(π || π_SFT)
- L_PG: clip 目标，用 importance ratio ρ_t = π/π_old
- KL 罚：防 π 偏离 SFT 过远
- GAE: 用 V 函数估计优势

## 数据流
SFT data -> [(x, y_chosen, y_rejected)] -> RM -> reward scores
RM + prompts -> PPO rollout -> (x, y, reward) -> update

## 考察点
- 三阶段各自的数据和损失
- KL 罚的作用（防 reward hacking）
- PPO 与 DPO 的 online/offline 区别


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from copy import deepcopy

class TinyLM(nn.Module):
    """简易语言模型用于 RLHF 演示。"""
    def __init__(self, vocab: int = 20, dim: int = 32) -> None:
        super().__init__()
        self.embed = nn.Embedding(vocab, dim)
        self.rnn = nn.GRU(dim, dim, batch_first=True)
        self.head = nn.Linear(dim, vocab, bias=False)
        self.value_head = nn.Linear(dim, 1)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        h = self.embed(tokens)
        out, _ = self.rnn(h)
        logits = self.head(out)
        return logits

    def get_logprobs(self, tokens: torch.Tensor) -> torch.Tensor:
        logits = self.forward(tokens)
        return F.log_softmax(logits, dim=-1)

    def get_value(self, tokens: torch.Tensor) -> torch.Tensor:
        h = self.embed(tokens)
        out, _ = self.rnn(h)
        return self.value_head(out[:, -1])

class RewardModel(nn.Module):
    """奖励模型：序列 -> 标量奖励。"""
    def __init__(self, vocab: int = 20, dim: int = 32) -> None:
        super().__init__()
        self.embed = nn.Embedding(vocab, dim)
        self.rnn = nn.GRU(dim, dim, batch_first=True)
        self.head = nn.Linear(dim, 1)

    def forward(self, tokens: torch.Tensor) -> torch.Tensor:
        h = self.embed(tokens)
        out, _ = self.rnn(h)
        return self.head(out[:, -1]).squeeze(-1)

class RLHFPipeline:
    def __init__(self, sft_model: nn.Module, reward_model: nn.Module, ref_model: nn.Module = None) -> None:
        self.policy = sft_model
        self.rm = reward_model
        self.ref = ref_model or deepcopy(sft_model)
        for p in self.rm.parameters(): p.requires_grad = False
        for p in self.ref.parameters(): p.requires_grad = False

    def compute_reward(self, input_ids: torch.Tensor) -> torch.Tensor:
        """RM 打分。"""
        with torch.no_grad():
            return self.rm(input_ids)

    def compute_kl_penalty(self, policy_logprobs: torch.Tensor, ref_logprobs: torch.Tensor) -> torch.Tensor:
        """KL(π || ref) token-wise sum。"""
        return (policy_logprobs - ref_logprobs).sum(dim=-1).mean()

    def rollout(self, prompt, max_new: int = 5) -> torch.Tensor:
        """用 policy 生成序列。"""
        tokens = list(prompt)
        with torch.no_grad():
            for _ in range(max_new):
                x = torch.tensor([tokens], dtype=torch.long)
                logprobs = self.policy.get_logprobs(x)
                nxt = torch.multinomial(torch.exp(logprobs[0, -1]), 1).item()
                tokens.append(nxt)
        return tokens

    def ppo_step(self, batch, epsilon: float = 0.2, beta: float = 0.04, entropy_coef: float = 0.01) -> torch.Tensor:
        """
        完整 PPO 更新 step:
        1. 计算 old_logprobs, ref_logprobs (no_grad)
        2. 前向 policy 得 new_logprobs, values
        3. GAE 计算优势
        4. clip 目标 + value loss + entropy + KL loss
        """
        prompts, responses, old_rewards = batch
        total_loss = 0.0
        for prompt, response, reward in zip(prompts, responses, old_rewards):
            tokens = torch.tensor([prompt + response], dtype=torch.long)
            with torch.no_grad():
                old_logprobs = self.policy.get_logprobs(tokens)
                ref_logprobs = self.ref.get_logprobs(tokens)
                value = self.policy.get_value(tokens)

            new_logprobs = self.policy.get_logprobs(tokens)
            ratio = torch.exp(new_logprobs - old_logprobs)
            advantage = reward - value.item()
            surr1 = ratio * advantage
            surr2 = torch.clamp(ratio, 1 - epsilon, 1 + epsilon) * advantage
            pg_loss = -torch.min(surr1, surr2).mean()

            kl = self.compute_kl_penalty(new_logprobs, ref_logprobs)
            entropy = -(torch.exp(new_logprobs) * new_logprobs).sum(dim=-1).mean()
            loss = pg_loss + beta * kl - entropy_coef * entropy
            total_loss += loss
        return total_loss / len(prompts)


In [ ]:
# ===== 测试验证 =====
torch.manual_seed(42)
vocab = 20
sft = TinyLM(vocab)
rm = RewardModel(vocab)
rlhf = RLHFPipeline(sft, rm)

tokens = torch.tensor([[1, 2, 3, 4]], dtype=torch.long)
reward = rlhf.compute_reward(tokens)
assert reward.shape == (1,), f"reward 形状错误: {reward.shape}"
print(f"✅ compute_reward: {reward.item():.4f}")

policy_lp = rlhf.policy.get_logprobs(tokens)
ref_lp = rlhf.ref.get_logprobs(tokens)
kl = rlhf.compute_kl_penalty(policy_lp, ref_lp)
assert kl.item() > -1e3, "KL 应有限"
print(f"✅ compute_kl_penalty: {kl.item():.4f}")

prompt = [1, 2, 3]
response = rlhf.rollout(prompt, max_new=3)
assert len(response) == len(prompt) + 3
print(f"✅ rollout: {prompt} -> {response}")

batch = ([prompt], [response[3:]], [0.5])
loss = rlhf.ppo_step(batch)
assert loss.item() > -1e3, "loss 应有限"
print(f"✅ ppo_step: loss={loss.item():.4f}")

loss.backward()
grad_count = sum(1 for p in rlhf.policy.parameters() if p.grad is not None)
assert grad_count > 0, "policy 应有梯度"
print(f"✅ 反向传播: {grad_count} params 有梯度")

for p in rlhf.rm.parameters():
    assert not p.requires_grad, "RM 应冻结"
for p in rlhf.ref.parameters():
    assert not p.requires_grad, "ref 应冻结"
print("✅ RM 和 ref 模型已冻结")
print("✅ 全部测试通过")
